In [1]:
import torch, os, sys
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR
from torch.utils.data import DataLoader
import time
import copy
import gc
from lib.helpers import generate_sample, pad_collate_fn, load_configs, get_data_loader, create_model, get_step_info
from pathlib import Path
from tomlkit import parse, dumps
from functools import partial
from tqdm import tqdm
import torchinfo
from data.datasets import SimpleStoriesBPEDataset
from datasets import load_dataset

LOSS_REGISTRY = {"CrossEntropyLoss": torch.nn.CrossEntropyLoss}
OPTIMIZER_REGISTRY = {"Adam": torch.optim.Adam, "SGD": torch.optim.SGD, "AdamW": torch.optim.AdamW}
SCHEDULER_REGISTRY = {"OneCycleLR": OneCycleLR, "Cosine": CosineAnnealingLR}


print("loading configs...")
model_cfg_path = Path("experiment/model.toml")
data_cfg_path = Path("experiment/data.toml")
train_cfg_path = Path("experiment/training.toml")

model_cfg = parse(model_cfg_path.read_text(encoding="utf-8"))
data_cfg = parse(data_cfg_path.read_text(encoding="utf-8"))
train_cfg = parse(train_cfg_path.read_text(encoding="utf-8"))

print("getting dataset")
data = load_dataset(data_cfg["dataset"])
dataset = SimpleStoriesBPEDataset(data[data_cfg["split"]], max_length=data_cfg["max_length"])

/Users/samsutherland/miniconda3/envs/transformers/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/Users/samsutherland/miniconda3/envs/transformers/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading configs...
getting dataset


In [4]:
len(dataset.tok.vocab)

1076

In [16]:
torch.set_default_dtype(torch.bfloat16)
model = Transformer(8, max_context=500).to(dtype=torch.bfloat16, device=device)
model.embedding.weight.data = model.embedding.weight.data.to(torch.bfloat16)
model.embedding.padding_idx = dataset.pad_id
criterion = torch.nn.CrossEntropyLoss(ignore_index=dataset.pad_id)

In [17]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
epochs = 3
save_dir = "checkpoints"
os.makedirs(save_dir, exist_ok=True)
best_loss = float("inf")
global_step = 0
loss_steps, loss_values = [], []
prompt_text = "The dog"

def generate_sample(prompt, n_words=15, max_new_tokens=60):
    model.eval()
    with torch.no_grad():
        ids0 = dataset.tok.encode(prompt)
        if len(ids0) == 0:
            ids0 = [dataset.pad_id]
        ids = torch.tensor(ids0, dtype=torch.long, device=device).unsqueeze(0)
        for _ in range(max_new_tokens):
            logits = model(ids)
            next_id = int(logits[0, -1].argmax(-1).item())
            if dataset.end_id is not None and next_id == dataset.end_id:
                break
            ids = torch.cat([ids, torch.tensor([[next_id]], device=device)], dim=1)
            if len(dataset.tok.decode(ids[0].tolist()).split()) >= n_words:
                break
    model.train()
    return " ".join(dataset.tok.decode(ids[0].tolist()).split()[:n_words])


for epoch in range(1, epochs + 1):
    model.train()
    pbar = tqdm(loader, desc=f"epoch {epoch}/{epochs}")
    epoch_loss, n_batches = 0.0, 0
    for batch in pbar:
        if global_step % 10 == 0:
            try:
                s = generate_sample(prompt_text, 15, 60)
                pbar.write(s)
            except Exception as e:
                pbar.write(f"[gen err: {e}]")
        if global_step % 500 == 0:
            torch.save(
                {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": global_step, "epoch": epoch},
                os.path.join(save_dir, f"ckpt_step_{global_step}.pt"),
            )
            torch.save(
                {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": global_step, "epoch": epoch},
                os.path.join(save_dir, "ckpt_latest.pt"),
            )
            torch.save({"steps": loss_steps, "losses": loss_values}, os.path.join(save_dir, "loss_history.pt"))
        x = batch["input_ids"].to(device, non_blocking=True)
        logits = model(x[:, :-1])
        loss = criterion(logits.reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        global_step += 1
        epoch_loss += loss.item()
        n_batches += 1
        loss_steps.append(global_step)
        loss_values.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    avg_loss = epoch_loss / max(1, n_batches)
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(
            {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": global_step, "epoch": epoch},
            os.path.join(save_dir, "ckpt_best.pt"),
        )
    torch.save({"steps": loss_steps, "losses": loss_values}, os.path.join(save_dir, "loss_history.pt"))

epoch 1/3:   0%|          | 0/88154 [00:00<?, ?it/s]

Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=83, pipe_handle=182)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=83, pipe_handle=196)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/samsutherland/miniconda3/envs/ai/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Users/samsutherland/miniconda3/envs/ai/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_mai

RuntimeError: DataLoader worker (pid(s) 32792, 32794) exited unexpectedly